# ST446 WT2026 - Assigment 1

**Task information:** In this assignment, you will analyse global COVID-19 pandemic data. You will progressively move from low-level distributed processing (Spark RDDs) to structured analytics (Spark DataFrames and Spark SQL) and finally to NoSQL analytics using Bigtable.

The assignment consists of **4 parts**, as follows:
* **Part A:** Spark RDD (4 questions)
* **Part B:** Spark DataFrames (3 questions)
* **Part C:** Spark SQL (2 questions)
* **Part D:** Bigtable (2 questions)

Your **overall goal** is not only to compute results, but to justify analytical choices, understand trade-offs between APIs, and reflect on how AI tools (if used) supported or influenced your reasoning.

<hr>

## Setup

In [10]:
!pip install pyspark

In [11]:
# Python libraries
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

# Spark libraries
from pyspark.sql import SparkSession
# Spark SQL data types
from pyspark.sql.types import *
# Spark SQL functions
import pyspark.sql.functions as sql_f

<hr>

## Data

Check the data source/provider and information about the dataset, especially metadata.

* Dataset: **COVID-19 Pandemic**
* Data source: Our World in Data
* Latest version (including metadata/column information): https://docs.owid.io/projects/covid/en/latest/dataset.html
* Project information: https://ourworldindata.org/coronavirus


In [5]:
# -----------------------------------------------
# Dataset download and Hadoop ingestion
# -----------------------------------------------
!wget https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
!hadoop fs -put -f owid-covid-data.csv /

--2026-03-02 22:54:14--  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98391483 (94M) [text/plain]
Saving to: ‘owid-covid-data.csv.1’

owid-covid-data.csv 100%[===================>]  93.83M   228MB/s    in 0.4s    

2026-03-02 22:54:15 (228 MB/s) - ‘owid-covid-data.csv.1’ saved [98391483/98391483]

/bin/bash: line 1: hadoop: command not found


In [12]:
# added by  me
#filename = "owid-covid-data.csv"


In [6]:
# -----------------------------------------------
# Check Hadoop masternode
# -----------------------------------------------
!hadoop fs -ls /

/bin/bash: line 1: hadoop: command not found


In [7]:
# -----------------------------------------------
# HDFS file path
# Adjust to match your cluster name and file path
# -----------------------------------------------
filename = "hdfs://st446-cluster-w04-m:8020/owid-covid-data.csv"

In [13]:
# added by me
# from pyspark.sql import SparkSession

# spark = SparkSession.builder \
#     .appName("ST446-Assignment1") \
#     .master("local[*]") \
#     .getOrCreate()

<hr>

## Part A — Spark RDDs (4 questions)

**Goal:** Understand distributed data processing, key/value transformations, aggregation, and ranking logic.

---

### Dataset loading & exploration

As usual in most analytical scenarios, you can **load the raw data into a RDD** and keep it unchanged throughout your analytical pipeline. For each specific task, this RDD can be loaded into a new RDD and subject to the required transformations/actions.

In [14]:
# Raw RDD
raw_rdd = spark.sparkContext.textFile(filename)
# Check header (column names)
header = raw_rdd.first()
columns = header.split(",")
# Print out column names
for i, col_name in enumerate(columns):
    print(f"Index: {i}: {col_name}")

Index: 0: iso_code
Index: 1: continent
Index: 2: location
Index: 3: date
Index: 4: total_cases
Index: 5: new_cases
Index: 6: new_cases_smoothed
Index: 7: total_deaths
Index: 8: new_deaths
Index: 9: new_deaths_smoothed
Index: 10: total_cases_per_million
Index: 11: new_cases_per_million
Index: 12: new_cases_smoothed_per_million
Index: 13: total_deaths_per_million
Index: 14: new_deaths_per_million
Index: 15: new_deaths_smoothed_per_million
Index: 16: reproduction_rate
Index: 17: icu_patients
Index: 18: icu_patients_per_million
Index: 19: hosp_patients
Index: 20: hosp_patients_per_million
Index: 21: weekly_icu_admissions
Index: 22: weekly_icu_admissions_per_million
Index: 23: weekly_hosp_admissions
Index: 24: weekly_hosp_admissions_per_million
Index: 25: total_tests
Index: 26: new_tests
Index: 27: total_tests_per_thousand
Index: 28: new_tests_per_thousand
Index: 29: new_tests_smoothed
Index: 30: new_tests_smoothed_per_thousand
Index: 31: positive_rate
Index: 32: tests_per_case
Index: 33: t

---

### Question 1 - Baseline RDD

Using the **raw RDD** loaded in the previous cell:
* create a reduced **baseline RDD** named ` covid_rdd` to answer all questions in Part A
* show three samples from the baseline RDD

The baseline RDD needs only a few columns:

```
# | Index    | Column             |
# | -------- | ------------------ |
# | 0        | iso_code           |
# | 1        | continent          |
# | 2        | location           |
# | 3        | date               |
# | 4        | total_cases        |
# | 5        | new_cases          |
# | 17       | icu_patients       |
# | 19       | hosp_patients      |
# | 34       | total_vaccinations |
# | 38       | new_vaccinations   |
# | 62       | population         |

```

**IMPORTANT:** OWID frequently adds new columns to the dataset. **You must verify column positions using the dataset metadata before hard-coding indices. Your code should not assume indices blindly without validating the schema**.

**OWID Dataset — baseline RDD assumptions:**
* `iso_code` refers to three-letter country codes (e.g. USA, FRA, DEU) or to prefix `OWID_` for non-country level aggregated data (e.g. continents like 'Europe' etc)
* Rows where `continent` is empty (or `None`) can also correspond to non-country aggregates (e.g. European Union, Internatioanl, income groups)
* `total_cases` and `total_vaccinations` are cumulative, so `max(total_cases)` gives final count
* Dates are in `YYYY-MM-DD` format (lexicographically sortable)

**Requirements and Constraints:**
* Verify column positions in the raw RDD before hard-coding indices
* Baseline RDD must be named `covid_rdd`
* Remove headers when generating the baseline RDD
* Map only the necessary columns from the raw RDD into `covid_rdd`

**Expected output:** the baseline RDD should have the following structure:

```
# | Index    | Column             |
# | -------- | ------------------ |
# | 0        | iso_code           |
# | 1        | continent          |
# | 2        | location           |
# | 3        | date               |
# | 4        | total_cases        |
# | 5        | new_cases          |
# | 6        | icu_patients       |
# | 7        | hosp_patients      |
# | 8        | total_vaccinations |
# | 9        | new_vaccinations   |
# | 10       | population         |
```

A sample should look like:
```
[('AFG',
  'Asia',
  'Afghanistan',
  '2020-01-05',
  '0',
  '0',
  '',
  '',
  '',
  '',
  '41128772')
```

In [18]:
## CODE
columns = [
    "iso_code", "continent", "location", "date",
    "total_cases", "new_cases", "icu_patients",
    "hosp_patients", "total_vaccinations",
    "new_vaccinations", "population"
]

# Dynamically find their indices from the header
col_indices = [columns.index(col) for col in columns]
print("Verified column indices:", list(zip(columns, col_indices)))

# Remove header and map to only the needed columns
covid_rdd = raw_rdd.filter(lambda line: line != header) \
    .map(lambda line: line.split(",")) \
    .map(lambda fields: tuple(fields[i] for i in col_indices))

#samples
covid_rdd.take(1)


Verified column indices: [('iso_code', 0), ('continent', 1), ('location', 2), ('date', 3), ('total_cases', 4), ('new_cases', 5), ('icu_patients', 6), ('hosp_patients', 7), ('total_vaccinations', 8), ('new_vaccinations', 9), ('population', 10)]


[('AFG',
  'Asia',
  'Afghanistan',
  '2020-01-05',
  '0',
  '0',
  '',
  '0',
  '0',
  '',
  '0.0')]

---

### Question 2: Hierarchical Aggregation (Continent -> Country)

Using the **baseline RDD**:
* Compute the total number of confirmed COVID-19 cases per continent level
* For each continent, identify the top 2 countries with the highest cumulative number of confirmed cases
* Show results from both RDDs

**Requirements and Constraints:**
* Use RDD transformations and actions only (no DataFrames or SQL)
* Filter out rows that correspond to aggregates and rows with missing continent
* Use the maximum cumulative total cases per country before aggregating at continent level

**Expected output:**
* One RDD with total cases per continent: `[(continent, total cases), ...]`
* One RDD with top 2 countries per continent by total cases, descending order: `[(continent, [(country1, total cases), (country2, total cases)]), ...]`
* Show results from both RDDs



In [19]:
# CODE
# CODE

# Indices in covid_rdd: 0=iso_code, 1=continent, 2=location, 3=date,
# 4=total_cases, 5=new_cases, 6=icu_patients, 7=hosp_patients,
# 8=total_vaccinations, 9=new_vaccinations, 10=population

# Step 1: Filter out aggregates (OWID_ prefix) and missing continents
filtered_rdd = covid_rdd.filter(
    lambda r: not r[0].startswith("OWID_") and r[1] != "" and r[1] != "None"
)

# Step 2: Get max cumulative total_cases per country
# Key by (continent, location), take max of total_cases
country_max = filtered_rdd \
    .filter(lambda r: r[4] != "") \
    .map(lambda r: ((r[1], r[2]), float(r[4]))) \
    .reduceByKey(max)
# country_max: ((continent, location), max_total_cases)

# --- RDD 1: Total cases per continent ---
continent_totals = country_max \
    .map(lambda r: (r[0][0], r[1])) \
    .reduceByKey(lambda a, b: a + b)

print("=== Total cases per continent ===")
for row in continent_totals.collect():
    print(row)

# --- RDD 2: Top 2 countries per continent ---
# Rekey by continent with (location, max_cases)
by_continent = country_max \
    .map(lambda r: (r[0][0], (r[0][1], r[1])))

# Group by continent, sort within each group, take top 2
top2_per_continent = by_continent \
    .groupByKey() \
    .mapValues(lambda countries: sorted(countries, key=lambda x: -x[1])[:2])

print("\n=== Top 2 countries per continent ===")
for row in top2_per_continent.collect():
    print(row)

=== Total cases per continent ===
('Europe', 252642589.0)
('North America', 124492666.0)
('South America', 68809418.0)
('Africa', 13145540.0)
('Asia', 301532347.0)
('Oceania', 15003352.0)

=== Top 2 countries per continent ===
('Europe', [('France', 38997490.0), ('Germany', 38437756.0)])
('North America', [('United States', 103436829.0), ('Mexico', 7619458.0)])
('South America', [('Brazil', 37511921.0), ('Argentina', 10101218.0)])
('Africa', [('South Africa', 4072765.0), ('Morocco', 1279115.0)])
('Asia', [('China', 99373219.0), ('India', 45041748.0)])
('Oceania', [('Australia', 11861161.0), ('New Zealand', 2639048.0)])


<hr>

### Question 3 – Temporal Growth Analysis

Using the **baseline RDD**:
* Restrict the dataset to the period March–June 2021
* Compute weekly total new COVID-19 cases per country
* For each continent, identify the single country with the highest week-over-week growth in new cases during this period
* Show the resulting RDD

**Requirements and Constraints:**
* Use RDD transformations and actions only (no DataFrames or SQL)
* Filter out missing or null values from the required columns
* Define week using ISO week number derived from the date. Week-over-week growth is defined as: `growth = (current_week_total − previous_week_total)`
* Growth should be computed within each country only

**Expected output:** RDD with `[(continent, (country, highest week-over-week growth)), ...]`

In [20]:
# CODE
# CODE
from datetime import datetime

# Step 1: Filter to March-June 2021, valid rows only
march_june = covid_rdd.filter(
    lambda r: r[1] != "" and r[1] != "None"
              and not r[0].startswith("OWID_")
              and r[3] >= "2021-03-01" and r[3] <= "2021-06-30"
              and r[5] != ""
)

# Step 2: Map to ((country, continent, iso_year_week), new_cases)
def get_iso_week(date_str):
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    iso = dt.isocalendar()
    return (iso[0], iso[1])  # (year, week_number)

weekly_cases = march_june \
    .map(lambda r: ((r[2], r[1], get_iso_week(r[3])), float(r[5]))) \
    .reduceByKey(lambda a, b: a + b)
# Result: ((country, continent, (year, week)), weekly_total_new_cases)

# Step 3: Rekey by (country, continent) and group weeks together
country_weeks = weekly_cases \
    .map(lambda r: ((r[0][0], r[0][1]), (r[0][2], r[1]))) \
    .groupByKey() \
    .mapValues(lambda weeks: sorted(weeks, key=lambda x: x[0]))
# Result: ((country, continent), [(week1, total), (week2, total), ...])

# Step 4: Compute max week-over-week growth per country
def max_growth(sorted_weeks):
    max_g = float('-inf')
    for i in range(1, len(sorted_weeks)):
        growth = sorted_weeks[i][1] - sorted_weeks[i-1][1]
        if growth > max_g:
            max_g = growth
    return max_g if max_g != float('-inf') else 0.0

country_growth = country_weeks \
    .filter(lambda r: len(r[1]) >= 2) \
    .map(lambda r: (r[0][1], (r[0][0], max_growth(r[1]))))
# Result: (continent, (country, max_growth))

# Step 5: For each continent, find the country with highest growth
top_growth = country_growth \
    .reduceByKey(lambda a, b: a if a[1] > b[1] else b)

print("=== Highest week-over-week growth per continent ===")
for row in top_growth.collect():
    print(row)

=== Highest week-over-week growth per continent ===
('Europe', ('France', 40845.0))
('North America', ('United States', 38267.0))
('South America', ('Brazil', 80556.0))
('Africa', ('South Africa', 32958.0))
('Asia', ('India', 742759.0))
('Oceania', ('Papua New Guinea', 782.0))


<hr>

### Question 4: Vaccination Intensity Ranking

Using the **baseline RDD**:
* Compute an RDD with the average daily number of new COVID-19 vaccinations per country.
* Return the top 10 countries by average daily vaccinations.

**Requirements & Constraints:**
* Use RDD transformations and actions only
* Use valid (not null/missing) daily new vaccinations
* Do not use cumulative vaccination columns
* Include only country-level records (exclude continents, world totals, and income groups)
* Include only countries with at least 50 days of valid reported vaccination data
* Round numeric outputs to two decimal places

**Expected output:** RDD with `[(country, average daily vaccinations), ...]`

In [21]:
# CODE
# CODE

# covid_rdd indices: 0=iso_code, 1=continent, 2=location, 3=date,
# 4=total_cases, 5=new_cases, 6=icu_patients, 7=hosp_patients,
# 8=total_vaccinations, 9=new_vaccinations, 10=population

# Step 1: Filter to country-level rows with valid new_vaccinations
vacc_rdd = covid_rdd.filter(
    lambda r: not r[0].startswith("OWID_")
              and r[1] != "" and r[1] != "None"
              and r[9] != ""
)

# Step 2: Map to (country, (new_vaccinations, 1)) for computing average
country_vacc = vacc_rdd \
    .map(lambda r: (r[2], (float(r[9]), 1))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
# Result: (country, (sum_new_vaccinations, count_of_days))

# Step 3: Filter countries with at least 50 valid days
country_vacc_50 = country_vacc.filter(lambda r: r[1][1] >= 50)

# Step 4: Compute average, round to 2 decimal places
country_avg = country_vacc_50 \
    .map(lambda r: (r[0], round(r[1][0] / r[1][1], 2)))

# Step 5: Sort descending and take top 10
top10 = country_avg.sortBy(lambda r: -r[1]).take(10)

print("=== Top 10 countries by average daily vaccinations ===")
for row in top10:
    print(row)

=== Top 10 countries by average daily vaccinations ===
('United States', 714.58)
('Brazil', 420.68)
('India', 319.73)
('Russia', 241.57)
('Mexico', 200.45)
('United Kingdom', 139.07)
('Germany', 137.02)
('Peru', 132.4)
('France', 132.38)
('Italy', 118.22)


<hr>

## Part B - Spark DataFrames (2 questions)

**Goal:** Understand structured analytics using transformations, aggregation, and ranking logic.

---

### Question 5: Data loading & baseline DataFrame

When loading data into Spark DataFrames, we usually define a mapping schema, based on `StructType` and `StructField`. This requires us to know the structure of the dataset and ensure that `StructField` names match the column names in the dataset; otherwise, NULL values are loaded. There are also other potential problems causing **column misalignment**, such as wrong separators and date formats.

For this part of the assignment, you must **create a baseline DataFrame** named `covid_df`:
* load the original dataset into a raw DataFrame (e.g., `raw_df`)
* filter out only the relevant columns (see table below) into a baseline DataFrame (e.g., `covid_df`)
* show three samples

**Requirements and Constraints:**
* instead of defining a schema, ask Spark to infer the schema for you. This is safer and avoids any potential column misalignment arising from the data
* note that we need **two additional columns** compared to the baseline RDD used in Part A.
* before attempting any DataFrames questions, print out some sample rows and check the inferred schema

**Expected output:** the baseline DataFrame should have this structure:

```
| Column name          | Description             |
| -------------------- | ----------------------- |
| `iso_code`           | Country codes           |
| `continent`          | Continent name          |
| `location`           | Country name            |
| `date`               | Observation date        |
| `total_cases`        | Cumulative COVID cases  |
| `new_cases`          | New cases               |
| `icu_patients`       | ICU patients            |
| `hosp_patients`      | Hospitalized patients   |
| `total_vaccinations` | Cumulative vaccinations |
| `new_vaccinations`   | Daily vaccinations      |
| `population`         | Total population        |
| `total_deaths`       | Total deaths            |
| `new_deaths`         | New deaths              |
| -------------------- | ----------------------- |
```

A sample will look like:

```
----------+----------+------------+----------+
|iso_code|continent|location |date      |total_cases|new_cases|icu_patients|hosp_patients|total_vaccinations|new_vaccinations|population|total_deaths|new_deaths|
+--------+---------+---------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|AUS     |Oceania  |Australia|2021-03-22|29196      |0        |1           |66           |312502            |30542           |26177410  |925         |0         |
```


In [22]:
# CODE
# CODE

# Load raw DataFrame with inferred schema
raw_df = spark.read.csv(filename, header=True, inferSchema=True)

# Select the needed columns (same as Part A + total_deaths, new_deaths)
covid_df = raw_df.select(
    "iso_code", "continent", "location", "date",
    "total_cases", "new_cases", "icu_patients", "hosp_patients",
    "total_vaccinations", "new_vaccinations", "population",
    "total_deaths", "new_deaths"
)

# Show three samples
covid_df.show(3)

# Check the inferred schema
covid_df.printSchema()

+--------+---------+-----------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|iso_code|continent|   location|      date|total_cases|new_cases|icu_patients|hosp_patients|total_vaccinations|new_vaccinations|population|total_deaths|new_deaths|
+--------+---------+-----------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|     AFG|     Asia|Afghanistan|2020-01-05|          0|        0|        NULL|         NULL|              NULL|            NULL|  41128772|           0|         0|
|     AFG|     Asia|Afghanistan|2020-01-06|          0|        0|        NULL|         NULL|              NULL|            NULL|  41128772|           0|         0|
|     AFG|     Asia|Afghanistan|2020-01-07|          0|        0|        NULL|         NULL|              NULL|            NULL|  41128772|           0|         0|
+--------+------

---

### Question 6: System Pressure Indicator (ICU aggregations)

Using the **baseline DataFrame**:
* Compute a DataFrame with the average ICU patient load per country
* Show the top 10 countries

**Requirements and Constraints:**
* Use only country-level rows (exclude all continent / world aggregates)
* Include only countries with at least 50 non-null ICU observations
* Order results in descending order

**Expected output:** DataFrame with
* `location`: country name
* `icu_obs_count`: count of valid ICU observations
* `avg_icu_patients`: computed average ICU patient load

In [23]:
# CODE

from pyspark.sql.functions import col, avg, count, round as spark_round

icu_top10 = covid_df \
    .filter(~col("iso_code").startswith("OWID_")) \
    .filter(col("continent").isNotNull()) \
    .filter(col("icu_patients").isNotNull()) \
    .groupBy("location") \
    .agg(
        count("icu_patients").alias("icu_obs_count"),
        spark_round(avg("icu_patients"), 2).alias("avg_icu_patients")
    ) \
    .filter(col("icu_obs_count") >= 50) \
    .orderBy(col("avg_icu_patients").desc()) \
    .limit(10)

icu_top10.show()

+--------------+-------------+----------------+
|      location|icu_obs_count|avg_icu_patients|
+--------------+-------------+----------------+
| United States|         1381|         7703.35|
|     Argentina|          647|         2934.58|
|        France|         1109|         2046.52|
|       Germany|         1194|         1771.24|
|         Spain|         1045|          1060.6|
|United Kingdom|          781|           900.5|
|  South Africa|          873|          803.65|
|         Chile|         1248|          795.66|
|         Japan|          203|          706.84|
|         Italy|         1627|          695.05|
+--------------+-------------+----------------+



---

### Question 7: Vaccination Momentum vs Population Size

Using the **baseline DataFrame**, for each country, compute:
* total number of vaccinations administered
* average daily vaccinations
* number of days with reported vaccination data
* `vaccinations_per_1000`, given by `(total vaccinations/population) * 1000`

**Requirements:**
* Restrict the dataset to country-level observations only (exclude continents and other aggregates)
* Consider only rows with valid `new_vaccinations` and `population` data
* Include only countries with at least 50 reporting days
* Round numeric outputs to two decimal places
* Order the results by`vaccinations_per_1000` in descending order

**Expected output:** DataFrame with:
* `location`
* `population`
* `total_vaccination`
* `avg_daily_vaccinations`
* `reporting_days`
* `vaccinations_per_1000`



In [24]:
# CODE
# CODE

from pyspark.sql.functions import col, avg, count, max as spark_max, first, round as spark_round

vacc_df = covid_df \
    .filter(~col("iso_code").startswith("OWID_")) \
    .filter(col("continent").isNotNull()) \
    .filter(col("new_vaccinations").isNotNull()) \
    .filter(col("population").isNotNull())

vacc_result = vacc_df \
    .groupBy("location") \
    .agg(
        first("population").alias("population"),
        spark_max("total_vaccinations").alias("total_vaccination"),
        spark_round(avg("new_vaccinations"), 2).alias("avg_daily_vaccinations"),
        count("new_vaccinations").alias("reporting_days")
    ) \
    .filter(col("reporting_days") >= 50) \
    .withColumn(
        "vaccinations_per_1000",
        spark_round((col("total_vaccination") / col("population")) * 1000, 2)
    ) \
    .select("location", "population", "total_vaccination",
            "avg_daily_vaccinations", "reporting_days", "vaccinations_per_1000") \
    .orderBy(col("vaccinations_per_1000").desc())

vacc_result.show()

+--------------------+----------+-----------------+----------------------+--------------+---------------------+
|            location|population|total_vaccination|avg_daily_vaccinations|reporting_days|vaccinations_per_1000|
+--------------------+----------+-----------------+----------------------+--------------+---------------------+
|                Cuba|  11212198|         45622024|              59634.13|           644|              4068.96|
|           Gibraltar|     32677|           118943|                501.27|           184|              3639.96|
|               Chile|  19603736|         62688847|              83562.57|           750|               3197.8|
|               Japan| 123951696|        383747738|              489532.5|           783|              3095.95|
|              Taiwan|  23893396|         67913330|             109593.57|           467|              2842.35|
|           Hong Kong|   7488863|         20999059|               23336.7|           893|              2

<hr>

## Part C: Spark SQL (2 questions)

**Goal:** Use high-level, intuitive SQL-like statements over DataFrame structured data to produce country-level summary indicators.


---

### Question 8: Pandemic Severity Summary

Using the **baseline DataFrame** as a Spark SQL table, create a **country-level summary table** (`severity_df`) that supports cross-country risk comparison. For each country, compute:
* the maximum cumulative number of confirmed COVID-19 cases
* the maximum cumulative number of confirmed COVID-19 deaths
* `case_fatality_ratio` (CFR), defined as `(maximum total deaths / maximum total cases) * 100`

**Requirements:**
* Restrict the analysis to country-level observations only
* Include only rows with valid `total_cases` and `total_deaths` data
* CFR is approximated using maximum cumulative values; this does not account for temporal lag between cases and deaths
* Round numeric outputs to two decimal places
* Order the table by `case_fatality_ratio` in descending order

**Expected output:** `severity_df` DataFrame with:
* `location`
* `max_total_cases`
* `max_total_deaths`
* `case_fatality_ratio`


In [26]:
# Register the baseline DataFrame as a Spark SQL table
covid_df.createOrReplaceTempView("covid")

severity_df = spark.sql("""
    SELECT
        location,
        MAX(total_cases) AS max_total_cases,
        MAX(total_deaths) AS max_total_deaths,
        ROUND(try_divide(MAX(total_deaths), MAX(total_cases)) * 100, 2) AS case_fatality_ratio
    FROM covid
    WHERE iso_code NOT LIKE 'OWID_%'
        AND continent IS NOT NULL
        AND total_cases IS NOT NULL
        AND total_deaths IS NOT NULL
    GROUP BY location
    ORDER BY case_fatality_ratio DESC
""")

severity_df.show()

+--------------------+---------------+----------------+-------------------+
|            location|max_total_cases|max_total_deaths|case_fatality_ratio|
+--------------------+---------------+----------------+-------------------+
|               Yemen|          11945|            2159|              18.07|
|               Sudan|          63993|            5046|               7.89|
|               Syria|          57423|            3163|               5.51|
|             Somalia|          27334|            1361|               4.98|
|                Peru|        4526977|          220975|               4.88|
|               Egypt|         516023|           24830|               4.81|
|              Mexico|        7619458|          334551|               4.39|
|Bosnia and Herzeg...|         403666|           16392|               4.06|
|             Liberia|           8090|             294|               3.63|
|         Afghanistan|         235214|            7998|                3.4|
|           

---

### Question 9: Health System & Vaccination Readiness

Using the **baseline DataFrame** as a Spark SQL table, create a **country-level summary table** (`health_vax_df`) combining indicators of healthcare system burden and vaccination progress. Such summary table would enable comparison of countries' readiness to manage COVID-19.

For each country, compute:
* the average number of hospitalised COVID-19 patients over time
* the average number of ICU COVID-19 patients over time
* vaccinations per 1,000 people (`vax_per_1k`), defined as `(maximum total vaccinations / population) * 1000`

**Requirements and Constraints:**
* Restrict the analysis to country-level observations only
* For each metric, use non-null observations; however, only include countries where all required columns are non-null in the final result
* Round all numeric outputs to two decimal places
* Order the results by `vax_per_1k` in descending order
* Remember that `total_vaccinations` is a cumulative variable

**Expected output:** `health_vax_df` DataFrame with:
* `location`
* `avg_hosp_patients`
* `avg_icu_patients`
* `vax_per_1k`


In [27]:
# CODE
# CODE

health_vax_df = spark.sql("""
    SELECT
        location,
        ROUND(AVG(hosp_patients), 2) AS avg_hosp_patients,
        ROUND(AVG(icu_patients), 2) AS avg_icu_patients,
        ROUND(try_divide(MAX(total_vaccinations), MAX(population)) * 1000, 2) AS vax_per_1k
    FROM covid
    WHERE iso_code NOT LIKE 'OWID_%'
        AND continent IS NOT NULL
    GROUP BY location
    HAVING avg_hosp_patients IS NOT NULL
        AND avg_icu_patients IS NOT NULL
        AND vax_per_1k IS NOT NULL
    ORDER BY vax_per_1k DESC
""")

health_vax_df.show()

+--------------+-----------------+----------------+----------+
|      location|avg_hosp_patients|avg_icu_patients|vax_per_1k|
+--------------+-----------------+----------------+----------+
|         Japan|         12171.04|          706.84|   3095.95|
|      Portugal|          1232.34|          184.79|   2756.81|
|       Belgium|          1698.76|          286.75|   2699.86|
|        Sweden|           807.32|          111.34|   2683.84|
|        Canada|          3576.44|          300.21|   2675.31|
|     Australia|          1370.03|           72.37|   2662.09|
|       Denmark|            351.6|           28.03|    2543.5|
|   South Korea|          3510.05|          307.32|   2502.09|
|         Italy|          8286.24|          695.05|   2449.44|
|       Finland|           379.49|           25.86|   2405.29|
|       Austria|          1199.32|          171.58|   2289.67|
|        France|         17463.67|         2046.52|   2278.22|
|   Netherlands|           607.95|           161.7|   2

<hr>

## PART D — Google Bigtable

**Goal:** Model a NoSQL (key/value pairs, in this case) table and run analytical queries.

Using the summary tables (`severity_df` and `health_vax_df`) that you computed in Part C, you will:
* Create a Bigtable instance and table.
* Define appropriate column families.
* Load data from summary tables into Bigtable.
* Run analytical queries on Bigtable rows.

---

### Bigtable setup and authentication

[This is based on Week02's seminar example]


In [ ]:
# -----------------------------------------------
# Ensure latest version of Bigtable GCP connector
# -----------------------------------------------
!pip install --upgrade google-cloud-bigtable

In [ ]:
# -----------------------------------------------
# Client library for authentication in GCP
# -----------------------------------------------
from oauth2client.client import GoogleCredentials

In [ ]:
# --------------------------------------------------------
# If Bigtable API is enabled, import necessary libraries
# --------------------------------------------------------
from google.cloud import bigtable
from google.cloud.bigtable import enums
from google.cloud.bigtable.column_family import MaxVersionsGCRule
import datetime

In [ ]:
# -----------------------------------------------
# Retrieve GCP credentials
# -----------------------------------------------
credentials = GoogleCredentials.get_application_default()
print(credentials)

In [ ]:
# -----------------------------------------------
# Bigtable client connection
# -----------------------------------------------
# REPLACE WITH YOUR GCP PROJECT ID AND DESIRED ZONE
PROJECT_ID = "st446-wt2025"       # Replace with your GCP project ID
ZONE = "europe-west2-c"           # Replace with your preferred zone (ideally, the same of Dataproc cluster)

# Instantiate a BigTable client
# admin=True is necessary for writing operations
client = bigtable.Client(project=PROJECT_ID, admin=True)

# Check connection by listing any instances running on Bigtable
instances = client.list_instances()
# Print out all instances; empty if no instances exist
for i in instances[0]:
    print (i.name)

---

### Question 10: Designing and Populating a Bigtable Serving Layer

Using the two summary tables (`severity_df` and `health_vax_df`), design and implement a **Bigtable schema to store country-level COVID-19 performance metrics**.

You must:
* Create a Bigtable instance (`covid-instance`)
* Create a table (`covid_summary`) with appropriate column families (see below)
* Insert data from both Spark SQL summary tables
* Verify that the data was correctly inserted by reading a few sample rows from your Bigtable table

You must design a **denormalised Bigtable schema** with:
* Each row representing one country, identified by the country name (`location`) as the row key
* The following column families:
    * `meta`, with structural country information (`continent` and `population`)
    * `severity`, with pandemic outcome metrics (`max_total_cases`, `max_total_deaths`, `case_fatality_ratio`)
    * `health_vax`, with health system and vaccination metrics (`avg_hosp_patients`, `avg_icu_patients`, `vax_per_1k`)
    
**Requirements & Constraints:**
* Do not create multiple tables (only one denormalized table)
* Do not store all columns in one family
* Do not use joins inside Bigtable (1). All joins must be performed in Spark before writing to Bigtable
* Remember that Bigtable stores numeric values as encoded bytes (`utf-8`)
* Each country must appear only once
* All data must only include country-level rows with complete data entries (no NULLs), consistent with previous tasks

(1) Bigtable is designed to be a high-scalable, distributed denormalised structure, based on *column families* (groups of columns frequently accessed together). Join operations are expensive (time consuming) and unpredictable at scale, so Bigtable duplicates data to guarantee fast, single-row access.

**Expected output:** samples read from Bigtable should look like:

```
Row key: Australia
severity: case_fatality_ratio -> 0.21
severity: max_total_cases -> 11861161
severity: max_total_deaths -> 25236
health_vax: avg_hosp_patients -> 1708.11
health_vax: avg_icu_patients -> 121.09
health_vax: vax_per_1k -> 2647.56
meta: continent -> Oceania
meta: population -> 26177410
```

In [ ]:
# CODE

# ================================================
# Step 1: Join severity_df and health_vax_df in Spark
# ================================================

# We need continent and population from covid_df for the meta family
# Get distinct country-level meta info
meta_df = covid_df.filter(
    ~sql_f.col("iso_code").startswith("OWID_")
).filter(
    sql_f.col("continent").isNotNull()
).groupBy("location").agg(
    sql_f.first("continent").alias("continent"),
    sql_f.max("population").alias("population")
)

# Join all three: severity_df + health_vax_df + meta_df
# Inner join ensures only countries with complete data across all tables
combined_df = severity_df \
    .join(health_vax_df, on="location", how="inner") \
    .join(meta_df, on="location", how="inner") \
    .filter(
        sql_f.col("max_total_cases").isNotNull() &
        sql_f.col("max_total_deaths").isNotNull() &
        sql_f.col("case_fatality_ratio").isNotNull() &
        sql_f.col("avg_hosp_patients").isNotNull() &
        sql_f.col("avg_icu_patients").isNotNull() &
        sql_f.col("vax_per_1k").isNotNull() &
        sql_f.col("continent").isNotNull() &
        sql_f.col("population").isNotNull()
    )

# Collect to driver for Bigtable insertion
rows = combined_df.collect()
print(f"Total countries to insert: {len(rows)}")

# ================================================
# Step 2: Create Bigtable instance and table
# ================================================

INSTANCE_ID = "covid-instance"
TABLE_ID = "covid_summary"

# Create instance
instance = client.instance(
    INSTANCE_ID,
    display_name="COVID Summary Instance",
    instance_type=enums.Instance.Type.DEVELOPMENT,
    labels={}
)

cluster = instance.cluster(
    "covid-cluster",
    location_id=ZONE,
    serve_nodes=0  # Development instance doesn't need serve_nodes
)

if not instance.exists():
    operation = instance.create(clusters=[cluster])
    operation.result(timeout=300)
    print(f"Instance {INSTANCE_ID} created.")
else:
    print(f"Instance {INSTANCE_ID} already exists.")

# Create table with 3 column families
table = instance.table(TABLE_ID)

if not table.exists():
    gc_rule = MaxVersionsGCRule(1)
    column_families = {
        "meta": gc_rule,
        "severity": gc_rule,
        "health_vax": gc_rule
    }
    table.create(column_families=column_families)
    print(f"Table {TABLE_ID} created with column families: meta, severity, health_vax")
else:
    print(f"Table {TABLE_ID} already exists.")

# ================================================
# Step 3: Insert data into Bigtable
# ================================================

for row in rows:
    row_key = row["location"].encode("utf-8")
    bt_row = table.direct_row(row_key)

    # meta family
    bt_row.set_cell("meta", "continent", str(row["continent"]).encode("utf-8"))
    bt_row.set_cell("meta", "population", str(row["population"]).encode("utf-8"))

    # severity family
    bt_row.set_cell("severity", "max_total_cases", str(row["max_total_cases"]).encode("utf-8"))
    bt_row.set_cell("severity", "max_total_deaths", str(row["max_total_deaths"]).encode("utf-8"))
    bt_row.set_cell("severity", "case_fatality_ratio", str(row["case_fatality_ratio"]).encode("utf-8"))

    # health_vax family
    bt_row.set_cell("health_vax", "avg_hosp_patients", str(row["avg_hosp_patients"]).encode("utf-8"))
    bt_row.set_cell("health_vax", "avg_icu_patients", str(row["avg_icu_patients"]).encode("utf-8"))
    bt_row.set_cell("health_vax", "vax_per_1k", str(row["vax_per_1k"]).encode("utf-8"))

    bt_row.commit()

print(f"Inserted {len(rows)} rows into Bigtable.")

# ================================================
# Step 4: Verify by reading sample rows
# ================================================

print("\n=== Sample rows from Bigtable ===\n")
count = 0
for row in table.read_rows():
    row.cells  # force read
    print(f"Row key: {row.row_key.decode('utf-8')}")
    for family, cols in row.cells.items():
        for col, cells in cols.items():
            val = cells[0].value.decode("utf-8")
            print(f"  {family}: {col.decode('utf-8')} -> {val}")
    print()
    count += 1
    if count >= 3:
        break

---

### Question 11: COVID-19 Normalised Impact Score

You were asked to **engineer a composite metric to rank countries** analytically.

Using the country summary Bigtable table created in Question 8, compute a **normalised impact score** for each country, defined as:

```
normalised_burden = ((avg_hosp_patients + avg_icu_patients) / population) * 100000
```

and

```
impact_score = normalised_burden − (vax_per_1k / 1000)
```

and return the top 20 countries with the lowest impact score, sorted in ascending order.

**Requirements & Constraints:**
* For each country row, extract the necessary columns from the relevant column families
* Skip rows with missing required values
* Output must contain exactly 20 rows
* Do not perform joins (Bigtable is denormalised).
* Recall that values are stored as bytes (`utf-8`) in Bigtable, so numeric conversion is necessary before performing calculations

**Expected output:** DataFrame with:
* `country` (or `location`)
* `population`
* `avg_hosp_patients`
* `avg_icu_patients`
* `normalised_burden`
* `vax_per_1k`
* `impact_score`


In [ ]:
# CODE
# Read all rows from Bigtable and compute impact score
results = []

for row in table.read_rows():
    row.cells  # force read
    try:
        country = row.row_key.decode("utf-8")

        # Extract values from column families, decode from bytes
        population = float(row.cells["meta"][b"population"][0].value.decode("utf-8"))
        avg_hosp = float(row.cells["health_vax"][b"avg_hosp_patients"][0].value.decode("utf-8"))
        avg_icu = float(row.cells["health_vax"][b"avg_icu_patients"][0].value.decode("utf-8"))
        vax_per_1k = float(row.cells["health_vax"][b"vax_per_1k"][0].value.decode("utf-8"))

        # Compute normalised burden and impact score
        normalised_burden = ((avg_hosp + avg_icu) / population) * 100000
        impact_score = normalised_burden - (vax_per_1k / 1000)

        results.append((
            country, population, avg_hosp, avg_icu,
            round(normalised_burden, 2), vax_per_1k, round(impact_score, 2)
        ))
    except (KeyError, ValueError):
        # Skip rows with missing required values
        continue

# Sort by impact score ascending and take top 20
results.sort(key=lambda x: x[6])
top20 = results[:20]

# Convert to DataFrame for display
impact_df = spark.createDataFrame(
    top20,
    schema=["country", "population", "avg_hosp_patients", "avg_icu_patients",
            "normalised_burden", "vax_per_1k", "impact_score"]
)

impact_df.show(20, truncate=False)

---

### FOLLOW-UP QUESTIONS:

Pick one of these questions and discuss on your report:

* **FQ1.** Explain why a country may have a high vaccination per 1,000 metric but still show high average ICU or hospital occupancy.
* **FQ2.** How does average ICU load correlate (visually or conceptually) with vaccination intensity?
* **FQ3.** Which continents dominate the top CFR rankings? What structural factors (age structure, reporting quality, health capacity) might explain this?
* **FQ4.** What are limitations of the impact score calculated in Question 11?